In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd '/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant'

!ls -la

Mounted at /content/drive
/content/drive/My Drive/Agentic AI Projects/adjuster_assistant
total 17
drwx------ 2 root root 4096 Feb 20 01:25 app
drwx------ 2 root root 4096 Feb 20 01:25 data
-rw------- 1 root root  183 Feb 20 21:21 .env
drwx------ 2 root root 4096 Feb 20 01:25 notebooks
drwx------ 2 root root 4096 Feb 20 01:25 tests


In [ ]:
!pip -q install openai chromadb pypdf python-dotenv
from dotenv import load_dotenv
import os

loaded = load_dotenv("/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant/.env")
print("dotenv loaded:", loaded)
print("key present:", bool(os.getenv("OPENAI_API_KEY")))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
import sys, os
sys.path.insert(0, "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant")

from app.rag.loaders import load_pdf_pages

pdf_path = "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant/data/sample/policies/DEC_CP_POL123_TX_v1.pdf"  # change if needed
print("file exists:", os.path.exists(pdf_path), "size:", os.path.getsize(pdf_path))

pages = load_pdf_pages(pdf_path)
print("num pages:", len(pages))
for p in pages[:3]:
    print("page", p.page, "chars:", len(p.text))
    print("preview:", repr(p.text[:200]))

file exists: True size: 38600
num pages: 1
page 1 chars: 1330
preview: 'COMMERCIAL  PROPERTY  POLICY  —  DECLARATIONS  PAGE  Policy  Number:  POL123  Named  Insured:  Acme  Wholesale,  LLC  Mailing  Address:  100  Market  St,  Dallas,  TX  75201  Location  of  Premises:  '


In [ ]:
import sys
REPO = "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(sys.path[0])

from app.rag.loaders import load_pdf_pages
from app.rag.embeddings import Embedder
from app.rag.vectorstore import ChromaStore
from app.rag.claims_chunking import chunk_claim_document
from app.rag.ingest_policy import ingest_policy_pdf
from app.rag.policy_chunking import chunk_policy_pages

print("✅ Core modules import OK")

/content/drive/My Drive/Agentic AI Projects/adjuster_assistant
✅ Core modules import OK


In [ ]:
#Ingest policy pdfs
from app.rag.ingest_policy import ingest_policy_pdf

REPO = "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant"
PERSIST_DIR = f"{REPO}/data/chroma_db"

pdf_path = f"{REPO}/data/sample/policies/CP00_10_BaseForm_POL123_TX_v1.pdf"
policy_md = {
    "policy_id": "POL123",
    "state": "TX",
    "effective_date": "2025-01-01",
    "expiration_date": "2026-01-01",
    "doc_version": "v1",
    "doc_type": "policy",
    "doc_id": "CP-00-10",
    "form_name": "CP 00 10",
    "endorsement": False,
    "priority": 1,
}

n = ingest_policy_pdf(pdf_path=pdf_path, policy_metadata=policy_md, persist_dir=PERSIST_DIR)
print("Policy chunks ingested:", n)

Policy chunks ingested: 4
